1. Import basic library

In [8]:
%pip install gower


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd
import numpy as np
import time
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.preprocessing import LabelEncoder
import gower
import warnings
warnings.filterwarnings('ignore')

2. Completa code for Attack I

================================================================================
ATTACK MODEL 3: k-NN BASED ATTACK EVALUATION PIPELINE
================================================================================

PIPELINE OVERVIEW:
-----------------
This pipeline evaluates the effectiveness of different distance metrics for 
attribute inference attacks using k-Nearest Neighbors (k-NN).

Unlike Attack Model 2 (which used meta-features + XGBoost), this approach uses
a simpler, more interpretable k-NN classifier that leverages the synthetic
dataset (SD) as the reference knowledge base.

ATTACK SCENARIO (Same as Attack Model 2):
----------------------------------------
The attacker has access to:
1. SYNTHETIC DATASET (SD): 10,000 synthetic samples WITH labels (diagnosis known)
2. TRAINING RECORDS (TR): Not directly used as training data in this attack!
   The attacker uses SD as the reference dataset instead.
3. VICTIM RECORDS (VR): 2,000 real records where diagnosis is HIDDEN
4. BASELINE RECORDS (BR/M2): 2,000 real records NOT in training set

ATTACK MECHANISM:
----------------
For each victim record, the attacker:
1. Computes distances to ALL synthetic records (using various distance metrics)
2. Finds the k nearest synthetic neighbors
3. Performs weighted majority vote (inverse distance weighting)
4. Predicts the diagnosis class based on neighbors' labels

The key intuition: If synthetic data preserves the structure of the real data,
then a victim record should have neighbors with the correct diagnosis.

WHAT THIS PIPELINE EVALUATES:
----------------------------
This pipeline systematically compares DIFFERENT DISTANCE METRICS to determine
which ones work best for the attack:

METRICS TESTED (13 total):
-------------------------
CONTINUOUS/NUMERIC METRICS (11):
  - euclidean, l2: Standard Euclidean distance
  - sqeuclidean: Squared Euclidean
  - manhattan, l1, cityblock: Manhattan distance
  - cosine: Cosine similarity (angular distance)
  - braycurtis: Bray-Curtis dissimilarity
  - correlation: 1 - Pearson correlation
  - canberra: Canberra distance (weighted absolute difference)
  - nan_euclidean: Euclidean with NaN handling

BINARY METRICS (7, but used conditionally):
  - dice: Dice coefficient
  - jaccard: Jaccard similarity
  - sokalmichener, sokalsneath, yule, russellrao, rogerstanimoto

SPECIAL METRIC (1):
  - GOWER: Mixed-type distance (handles categorical + numeric data)

Additionally, the pipeline tests different k VALUES (39 values):
  Fine-grained: 1-15 (every integer)
  Medium: 17, 19, 21, 23, 25, 27, 29, 31, 35, 39, 43, 47
  Large: 51, 59, 67, 75, 83, 99, 115, 131, 147, 163, 179, 195

KEY EVALUATION METRICS:
----------------------
For each (metric, k) combination:
  - VR Accuracy: Attack success on victim records (should be high)
  - BR Accuracy: Baseline on test records (should be lower)
  - Delta = |VR Accuracy - BR Accuracy|: Information leakage indicator

SATURATION ANALYSIS:
-------------------
For each distance metric, we identify the SATURATION POINT:
  - The smallest k where adding more neighbors doesn't improve accuracy
  - Defined as: improvement < 0.005 over 3 consecutive k values
  - Helps select optimal k without overfitting

OUTPUTS:
--------
1. Complete results table (CSV):
   - All metrics × all k values
   - VR accuracy, BR accuracy, delta

2. Saturation analysis (CSV):
   - Saturation k for each metric
   - Maximum accuracy achieved
   - Minimum delta (privacy leakage measure)

3. Visualization Plots:
   a) VR Accuracy vs k (top 10 metrics)
   b) BR Accuracy vs k (top 10 metrics)
   c) Delta vs k with green (5%) and yellow (10%) thresholds
   d) Saturation analysis bar chart
   e) Heatmap of accuracies across metrics × k

INTERPRETATION GUIDELINES:
-------------------------
SUCCESSFUL ATTACK:
  - VR accuracy significantly > BR accuracy
  - Delta > 0.05 (green threshold)
  - Top metrics achieve VR accuracy > 0.40-0.50

MODERATE LEAKAGE:
  - Delta between 0.03 and 0.05
  - Attack works but limited effectiveness

GOOD PRIVACY:
  - Delta < 0.03
  - VR accuracy close to BR accuracy
  - Both near random baseline (0.25)

DATA PREPROCESSING:
------------------
This pipeline applies:
1. City name → Geographic distance (km from Puglia centroid)
2. One-hot encoding for categorical variables
3. Label encoding for non-Gower metrics
4. Special handling for binary metrics (boolean conversion)

COMPARISON WITH ATTACK MODEL 2:
-------------------------------
Attack Model 2 (XGBoost + meta-features): More complex, potentially higher accuracy
Attack Model 3 (k-NN + distance metrics): Simpler, interpretable, baseline comparison

================================================================================

In [2]:
#download data in folder work/data
!../sspcloud/download_data_all.sh

Executing the download_data.sh script
]11;?\mc: Configuration written to `/home/onyxia/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/home/onyxia/.mc/share`.
mc: Initialized share uploads `/home/onyxia/.mc/share/uploads.json` file.
mc: Initialized share downloads `/home/onyxia/.mc/share/downloads.json` file.
Added `s3sspcloud` successfully.

== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-05-25 09:26:25 +0000 UTC. Renew soon to avoid outage.

...lation.csv: 8.30 MiB / 8.30 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 47.83 MiB/s 0s
== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-05-25 09:26:25 +0000 UTC. Renew soon to avoid outage.

...ISS_X2.csv: 3.50 MiB / 3.50 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 20.07 MiB/s 0s
== WARN: `minio.lab.sspcloud.fr` certificate will expire in 2026-05-25 09:26:25 +0000 UTC. Renew soon to avoid outage.

..._CTGAN.csv: 2.15 MiB / 2.15 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 14.47 MiB/s 0s
== WARN: `mini

In [14]:
# ==========================================================
# CONFIGURATION
# ==========================================================

class Config:
    # Randomization settings
    R = 'Y'              # 'Y' for randomizzazione (RY), 'N' for no randomizzazione (RN)
    PR = 40              # Randomization percentage: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE = 'CTGAN'  # CTGAN, TVAE, XGBoost_O0
    
    # Derived config name
    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'
    
    # Base paths
    BASE_PATH = rf"/home/onyxia/work/data/"
    
    # Input paths
    INPUT_REAL = f'{BASE_PATH}/Step1/Output'
    INPUT_SYNTH = f'{BASE_PATH}/Step2/Output'
    INPUT_COMUNI = f'{BASE_PATH}/Step4/Input/gi_comuni.csv'
    
    # Output paths
    OUTPUT_REPORT = f'{BASE_PATH}/Step4/Output/Report'
    OUTPUT_PLOT = f'{BASE_PATH}/Step4/Output/Plot'
    
    # Extended k values up to ~200
    K_VALUES = [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,
        17, 19, 21, 23, 25, 27, 29, 31, 35, 39, 43, 47,
        51, 59, 67, 75, 83, 99, 115, 131, 147, 163, 179, 195
    ]
    
    # All pairwise metrics
    METRICS = [
        'l2', 'sqeuclidean', 'manhattan', 'l1', 'cityblock',
        'cosine', 'braycurtis', 'nan_euclidean', 'correlation', 
        'hamming', 'euclidean', 'canberra'
    ]
    
    BINARY_METRICS = ['sokalmichener', 'sokalsneath', 'yule', 'russellrao', 'dice', 'rogerstanimoto', 'jaccard']

SYNTHESIZER_TYPES = ['CTGAN', 'TVAE', 'XGBoost_O0']

# ==========================================================
# COLOR PALETTE (13 colors for 13 models)
# ==========================================================

COLOR_PALETTE = [
    '#2E86AB',  # Blue
    '#A23B72',  # Purple
    '#F18F01',  # Orange
    '#C73E1D',  # Red
    '#6A994E',  # Green
    '#BC4A6C',  # Pink
    '#1C7C54',  # Dark Green
    '#D62828',  # Bright Red
    '#003D5B',  # Navy
    '#E09F3E',  # Gold
    '#7B2CBF',  # Violet
    '#F72585',  # Hot Pink
    '#4CC9F0',  # Cyan
]

MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'H', 'd', 'p']

LINESTYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]

# ==========================================================
# SHARED STYLE SETUP
# ==========================================================

def setup_plot_style():
    """White background, no titles, clean grid."""
    plt.rcParams.update({
        'figure.facecolor':    'white',
        'axes.facecolor':      'white',
        'savefig.facecolor':   'white',
        'font.size':           12,
        'axes.labelsize':      14,
        'axes.titlesize':      16,
        'xtick.labelsize':     12,
        'ytick.labelsize':     12,
        'legend.fontsize':     11,
        'axes.grid':           True,
        'grid.alpha':          0.3,
        'grid.linestyle':      '--',
        # EPS-safe fonts
        'ps.useafm':           True,
        'pdf.use14corefonts':  True,
        'text.usetex':         False,
    })


# ==========================================================
# BUILD FIXED ORDERING (shared across all synthesizers)
# Uses REPORTS (not saturation files) to get max_acc_real
# ==========================================================

def build_fixed_model_order_from_reports(all_reports, model_col, top_n=13):
    """
    Use complete reports (with data for each k) to compute
    max_acc_real per model across all synthesizers.
    """
    combined = pd.concat(all_reports, ignore_index=True)
    
    # Calculate max acc_real per model
    max_per_model = combined.groupby(model_col)['acc_real'].max()
    max_per_model = max_per_model.sort_values(ascending=False)
    
    if top_n is not None and top_n < len(max_per_model):
        top_models = max_per_model.head(top_n).index.tolist()
    else:
        top_models = max_per_model.index.tolist()
    
    # Clean model labels for legend
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower').replace('GOWER_WEIGHTED', 'Gower')
                    for m in top_models]
    
    return top_models, model_labels


# ==========================================================
# SAVE SHARED LEGEND (called once after order is fixed)
# ==========================================================

def save_shared_legend(top_models, model_labels, cfg, plot_path):
    """
    Build a dummy figure with one line per model (using the fixed
    colors/markers) purely to extract handles, then save the legend
    as a standalone EPS + JPEG file named without any synthesizer tag.
    """
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(1, 1))

    for idx, label in enumerate(model_labels):
        ax.plot(
            [], [],
            marker          = MARKERS[idx % len(MARKERS)],
            linestyle       = LINESTYLES[idx % len(LINESTYLES)],
            color           = COLOR_PALETTE[idx],
            label           = label,
            linewidth       = 2,
            markersize      = 6,
            markeredgecolor = 'black',
            markeredgewidth = 0.8,
        )

    handles, labels = ax.get_legend_handles_labels()
    plt.close(fig)

    # Calculate number of columns based on number of models
    n_models = len(model_labels)
    n_cols = min(7, n_models)
    
    fig_leg = plt.figure(figsize=(14, 1.5))
    fig_leg.legend(
        handles, labels,
        loc='center',
        ncol=n_cols,
        frameon=False,
        fontsize=10,
        handlelength=2.0,
        columnspacing=1.2,
    )
    plt.axis('off')

    legend_eps  = plot_path / f'legend_all_metrics_{cfg.RANDOMIZATION_LABEL}.eps'
    legend_jpeg = legend_eps.with_suffix('.jpeg')

    plt.savefig(legend_eps,  format='eps',  dpi=300,
                bbox_inches='tight', facecolor='white')
    plt.savefig(legend_jpeg, format='jpeg', dpi=150,
                bbox_inches='tight', facecolor='white')
    plt.close(fig_leg)

    print(f'  ✓ SHARED LEGEND EPS  → {legend_eps.name}')
    print(f'  ✓ SHARED LEGEND JPEG → {legend_jpeg.name}')


# ==========================================================
# PLOT: VR and BR accuracy vs k (fixed model order)
# ==========================================================

def plot_top_models_accuracy(results_df, cfg, plot_path,
                             top_models, model_labels):
    """
    Two separate EPS+JPEG plots (VR accuracy, BR accuracy).
    Uses the pre-computed fixed top_models / model_labels so that
    colors and markers are identical across all synthesizers.
    NO LEGEND in individual plots.
    """
    setup_plot_style()

    results_model_col = 'model' if 'model' in results_df.columns else 'modello'

    for metric, ylabel, suffix in [
        ('acc_real',   'TVR Accuracy', 'TVR'),
        ('acc_unseen', 'BVR Accuracy', 'BVR'),
    ]:
        fig, ax = plt.subplots(figsize=(11, 7))
        ax.set_facecolor('white')

        for idx, model in enumerate(top_models):
            # Check if model exists in this synthesizer's results
            if model not in results_df[results_model_col].values:
                continue
                
            df_m = (results_df[results_df[results_model_col] == model]
                    .sort_values('k'))
            
            if len(df_m) == 0:
                continue
                
            ax.plot(
                df_m['k'],
                df_m[metric],
                marker          = MARKERS[idx % len(MARKERS)],
                linestyle       = LINESTYLES[idx % len(LINESTYLES)],
                color           = COLOR_PALETTE[idx],
                linewidth       = 2,
                markersize      = 6,
                markeredgecolor = 'black',
                markeredgewidth = 0.8,
            )

        ax.set_xlabel(r'$k$', fontsize=14, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0.25, top=0.55)
        ax.grid(True, alpha=0.3, linestyle='--')

        # NO LEGEND in individual plots
        # Legend is saved separately as shared file

        plt.tight_layout()
        out_eps  = (plot_path /
                    f'accuracy_vs_k_{suffix}_{cfg.SYNTHESIZER_TYPE}'
                    f'_{cfg.RANDOMIZATION_LABEL}.eps')
        out_jpeg = out_eps.with_suffix('.jpeg')

        plt.savefig(out_eps,  format='eps',  dpi=300,
                    bbox_inches='tight', facecolor='white')
        plt.savefig(out_jpeg, format='jpeg', dpi=150,
                    bbox_inches='tight', facecolor='white')
        plt.close()

        print(f'  ✓ EPS  → {out_eps.name}')
        print(f'  ✓ JPEG → {out_jpeg.name}')


# ==========================================================
# DATA LOADING FUNCTIONS (from original code)
# ==========================================================

def load_data(cfg):
    # Build file paths
    real_path = rf'{cfg.INPUT_REAL}/real_data_datasetM10_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2.csv'
    unseen_path = rf'{cfg.INPUT_REAL}/real_data_datasetM2_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2.csv'
    synth_path = rf'{cfg.INPUT_SYNTH}/synthetic_data_datasetM10_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2_{cfg.SYNTHESIZER_TYPE}.csv'
    
    print(f"Loading real data from: {real_path}")
    print(f"Loading unseen data from: {unseen_path}")
    print(f"Loading synth data from: {synth_path}")
    
    real = pd.read_csv(real_path)
    unseen = pd.read_csv(unseen_path)
    synth = pd.read_csv(synth_path)
    
    # List of columns to drop (only those that exist in the dataframe)
    cols_to_drop = ['id', 'score1', 'score2', 'bin1', 'bin2', 'raw_class', 
                    'age_norm', 'activity_norm', 'predisposition_norm', 
                    'birth_year', 'birth_month', 'birth_day', 
                    'municipality_birth', 'birth_dayofyear']
    
    # Drop only columns that exist
    real_cols_to_drop = [col for col in cols_to_drop if col in real.columns]
    unseen_cols_to_drop = [col for col in cols_to_drop if col in unseen.columns]
    synth_cols_to_drop = [col for col in cols_to_drop if col in synth.columns]
    
    if real_cols_to_drop:
        real = real.drop(columns=real_cols_to_drop)
    if unseen_cols_to_drop:
        unseen = unseen.drop(columns=unseen_cols_to_drop)
    if synth_cols_to_drop:
        synth = synth.drop(columns=synth_cols_to_drop)
    
    return real, unseen, synth


def load_city_data(cfg):
    import unicodedata
    def normalize(text):
        if pd.isna(text):
            return text
        text = str(text).strip().upper()
        return unicodedata.normalize('NFKD', text).encode('ascii', errors='ignore').decode('utf-8')
    
    cities = pd.read_csv(cfg.INPUT_COMUNI, sep=";")
    cities['den_norm'] = cities['denominazione_ita'].apply(normalize)
    cities = cities[~((cities['den_norm'] == "CASTRO") & (cities['sigla_provincia'] != "LE"))]
    return cities.drop_duplicates(subset='den_norm')


def distance_from_puglia(city_name, cities):
    import unicodedata
    def normalize(text):
        if pd.isna(text):
            return text
        text = str(text).strip().upper()
        return unicodedata.normalize('NFKD', text).encode('ascii', errors='ignore').decode('utf-8')
    
    city_norm = normalize(city_name)
    row = cities[cities['den_norm'] == city_norm]
    if row.empty:
        return np.nan
    
    lat = float(str(row.iloc[0]['lat']).replace(',', '.'))
    lon = float(str(row.iloc[0]['lon']).replace(',', '.'))
    lat_ctr, lon_ctr = 41.25, 16.25
    
    R = 6371
    lat_rad, lon_rad = np.radians(lat), np.radians(lon)
    lat_ctr_rad, lon_ctr_rad = np.radians(lat_ctr), np.radians(lon_ctr)
    
    a = np.sin((lat_rad - lat_ctr_rad)/2)**2 + np.cos(lat_ctr_rad) * np.cos(lat_rad) * np.sin((lon_rad - lon_ctr_rad)/2)**2
    return int(round(R * 2 * np.arcsin(np.sqrt(a))))


def transform_cities(df, cities):
    df = df.copy()
    df['municipality_residence'] = df['municipality_residence'].apply(lambda x: distance_from_puglia(x, cities))
    for col in ['physical_activity', 'genetic_predisposition']:
        df[col] = pd.to_numeric(df[col].replace('Missing', 0), errors='coerce')
    return df


# ==========================================================
# EVALUATION FUNCTIONS
# ==========================================================

def prepare_for_gower(df):
    df = df.copy()
    df.replace('Missing', np.nan, inplace=True)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(float)
        else:
            df[col] = df[col].astype(object)
    return df


def majority_vote_weighted(y_neighbors, distances, epsilon=1e-5):
    weights = 1 / (distances + epsilon)
    classes = np.unique(y_neighbors)
    class_weights = np.array([weights[y_neighbors == c].sum() for c in classes])
    
    if len(class_weights) == 0 or np.all(np.isnan(class_weights)):
        return np.random.choice(y_neighbors), False
    
    max_weight = np.nanmax(class_weights)
    candidates = classes[class_weights == max_weight]
    return np.random.choice(candidates), len(candidates) > 1


def evaluate_knn(X_train, y_train, X_test, y_test, k_values, metric='gower', is_binary=False):
    results = []
    
    if metric == 'gower':
        X_train_prep = prepare_for_gower(X_train)
        X_test_prep = prepare_for_gower(X_test)
        dist_matrix = gower.gower_matrix(X_test_prep, X_train_prep)
    else:
        if is_binary:
            X_test_input = X_test.astype(bool)
            X_train_input = X_train.astype(bool)
        else:
            X_test_input = X_test.values
            X_train_input = X_train.values
        dist_matrix = pairwise_distances(X_test_input, X_train_input, metric=metric)
    
    for k in k_values:
        if k > len(X_train):
            continue
        indices = np.argsort(dist_matrix, axis=1)[:, :k]
        predictions = []
        
        for i in range(len(X_test)):
            neighbor_vals = y_train.iloc[indices[i]].values
            neighbor_dists = dist_matrix[i, indices[i]]
            pred, _ = majority_vote_weighted(neighbor_vals, neighbor_dists)
            predictions.append(pred)
        
        results.append({'k': k, 'accuracy': accuracy_score(y_test, predictions)})
    
    return results


# ==========================================================
# MAIN PIPELINE (modified to save reports for all synthesizers)
# ==========================================================

def run_pipeline_for_synthesizer(cfg, synth_type):
    """Run evaluation for a single synthesizer and save results"""
    cfg.SYNTHESIZER_TYPE = synth_type
    cfg.RANDOMIZATION_LABEL = f'RY_PR{cfg.PR}' if cfg.R == 'Y' else 'RN_PR0'
    
    print(f"\n{'='*60}")
    print(f"Processing synthesizer: {synth_type}")
    print(f"{'='*60}")
    
    # Create output directories
    report_path = Path(cfg.OUTPUT_REPORT)
    report_path.mkdir(parents=True, exist_ok=True)
    
    # Load and prepare data
    df_real, df_unseen, df_synth = load_data(cfg)
    cities = load_city_data(cfg)
    
    df_real = transform_cities(df_real, cities)
    df_unseen = transform_cities(df_unseen, cities)
    df_synth = transform_cities(df_synth, cities)
    
    X_synth, y_synth = df_synth.drop(columns=['diagnosis']), df_synth['diagnosis']
    X_real, y_real = df_real.drop(columns=['diagnosis']), df_real['diagnosis']
    X_unseen, y_unseen = df_unseen.drop(columns=['diagnosis']), df_unseen['diagnosis']
    
    # Label encoding for non-Gower metrics
    encoders = {}
    for col in X_synth.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_synth[col] = le.fit_transform(X_synth[col].astype(str))
        encoders[col] = le
    
    for df in [X_real, X_unseen]:
        for col in encoders:
            df[col] = df[col].map(lambda x: encoders[col].transform([str(x)])[0] if str(x) in encoders[col].classes_ else -1)
    
    # Storage for all results
    all_results = []
    
    # GOWER evaluation
    print("Evaluating GOWER...")
    gower_real = evaluate_knn(X_synth, y_synth, X_real, y_real, cfg.K_VALUES, metric='gower')
    gower_unseen = evaluate_knn(X_synth, y_synth, X_unseen, y_unseen, cfg.K_VALUES, metric='gower')
    
    gower_dict_real = {r['k']: r['accuracy'] for r in gower_real}
    gower_dict_unseen = {r['k']: r['accuracy'] for r in gower_unseen}
    
    for k in cfg.K_VALUES:
        if k in gower_dict_real and k in gower_dict_unseen:
            acc_real = gower_dict_real[k]
            acc_unseen = gower_dict_unseen[k]
            delta = abs(acc_real - acc_unseen)
            all_results.append({
                'model': 'GOWER',
                'k': k,
                'acc_real': acc_real,
                'acc_unseen': acc_unseen,
                'delta': delta
            })
    
    # Multi-metric evaluation
    for metric in cfg.METRICS:
        print(f"Evaluating {metric}...")
        is_binary = metric in cfg.BINARY_METRICS
        
        real_res = evaluate_knn(X_synth, y_synth, X_real, y_real, cfg.K_VALUES, metric=metric, is_binary=is_binary)
        unseen_res = evaluate_knn(X_synth, y_synth, X_unseen, y_unseen, cfg.K_VALUES, metric=metric, is_binary=is_binary)
        
        real_dict = {r['k']: r['accuracy'] for r in real_res}
        unseen_dict = {r['k']: r['accuracy'] for r in unseen_res}
        
        for k in cfg.K_VALUES:
            if k in real_dict and k in unseen_dict:
                acc_real = real_dict[k]
                acc_unseen = unseen_dict[k]
                delta = abs(acc_real - acc_unseen)
                all_results.append({
                    'model': f'MM_{metric}',
                    'k': k,
                    'acc_real': acc_real,
                    'acc_unseen': acc_unseen,
                    'delta': delta
                })
    
    # Create final dataframe
    results_df = pd.DataFrame(all_results)
    
    # Save CSV results
    results_df.to_csv(report_path / f'report_unificato_completo_{cfg.RANDOMIZATION_LABEL}_{synth_type}.csv', index=False)
    
    print(f"  ✓ Results saved for {synth_type}")
    
    return results_df


# ==========================================================
# MAIN PLOTTING FUNCTION
# ==========================================================

def generate_plots_only():
    """
    1. Run evaluation for all synthesizers to get report CSVs
    2. Load report CSVs to build the single fixed model ordering
    3. Save one shared legend file
    4. Generate plots for each synthesizer using fixed ordering
    """
    cfg = Config()
    
    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path   = Path(cfg.OUTPUT_PLOT)
    report_path.mkdir(parents=True, exist_ok=True)
    plot_path.mkdir(parents=True, exist_ok=True)
    
    print(f"Pipeline started")
    print(f"  Randomization: {cfg.RANDOMIZATION_LABEL}")
    print(f"  K values: {len(cfg.K_VALUES)} (max {max(cfg.K_VALUES)})")
    print(f"  Metrics: {len(cfg.METRICS)} + GOWER")
    print(f"  Report output: {report_path}")
    print(f"  Plot output: {plot_path}")
    print("-" * 60)
    
    # ----------------------------------------------------------
    # PASS 1: Run evaluation for all synthesizers
    # ----------------------------------------------------------
    all_reports = []
    loaded_data = {}
    
    for synth in SYNTHESIZER_TYPES:
        results_df = run_pipeline_for_synthesizer(cfg, synth)
        all_reports.append(results_df)
        loaded_data[synth] = results_df
    
    # ----------------------------------------------------------
    # PASS 2: Build fixed model ordering from all reports
    # ----------------------------------------------------------
    print("\n" + "=" * 60)
    print("Building fixed model ordering from all synthesizers")
    print("=" * 60)
    
    model_col = 'model'
    top_models, model_labels = build_fixed_model_order_from_reports(
        all_reports, model_col, top_n=13
    )
    
    print(f'\nFixed model order ({len(top_models)} models):')
    for i, (m, l) in enumerate(zip(top_models, model_labels)):
        print(f'  {i+1:2d}. {l}  ({m})')
    
    # ----------------------------------------------------------
    # PASS 3: Save shared legend
    # ----------------------------------------------------------
    print()
    save_shared_legend(top_models, model_labels, cfg, plot_path)
    
    # ----------------------------------------------------------
    # PASS 4: Generate plots for each synthesizer
    # ----------------------------------------------------------
    print()
    print("=" * 60)
    print("Generating plots with fixed legend")
    print("=" * 60)
    
    for synth, results_df in loaded_data.items():
        cfg.SYNTHESIZER_TYPE = synth
        print(f'\n--- {synth} ---')
        plot_top_models_accuracy(
            results_df, cfg, plot_path, top_models, model_labels
        )
    
    print()
    print(f"All EPS + JPEG saved to: {plot_path}")
    print("=" * 60)
    
    return loaded_data


# ==========================================================
# ENTRY POINT
# ==========================================================

if __name__ == '__main__':
    start = time.time()
    all_results = generate_plots_only()
    print(f"\nTotal execution time: {time.time() - start:.1f}s")

Pipeline started
  Randomization: RY_PR40
  K values: 39 (max 195)
  Metrics: 12 + GOWER
  Report output: /home/onyxia/work/data/Step4/Output/Report
  Plot output: /home/onyxia/work/data/Step4/Output/Plot
------------------------------------------------------------

Processing synthesizer: CTGAN
Loading real data from: /home/onyxia/work/data//Step1/Output/real_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2.csv
Loading unseen data from: /home/onyxia/work/data//Step1/Output/real_data_datasetM2_TH20_RY_PR40_4CAT_MISS_X2.csv
Loading synth data from: /home/onyxia/work/data//Step2/Output/synthetic_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2_CTGAN.csv


FileNotFoundError: [Errno 2] No such file or directory: '/home/onyxia/work/data//Step4/Input/gi_comuni.csv'

In [ ]:
# ==========================================================
# ONLY PLOTS - NO CALCULATIONS - NO TITLES - EPS + JPEG FORMAT
# Simplified version - NO COMUNI / NO CITY DISTANCE
# For ONYXIA environment
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
import time
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import pairwise_distances, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# ==========================================================
# GOWER DISTANCE IMPLEMENTATION (no external library needed)
# ==========================================================

def prepare_for_gower(df):
    df = df.copy()
    df.replace('Missing', np.nan, inplace=True)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].astype(float)
        else:
            df[col] = df[col].astype(object)
    return df

def gower_distance_matrix(X_train, X_test):
    """
    Compute Gower distance matrix between test and train sets.
    Gower distance handles mixed numeric and categorical data.
    """
    X_train = X_train.reset_index(drop=True)
    X_test = X_test.reset_index(drop=True)
    
    n_train = len(X_train)
    n_test = len(X_test)
    dist_matrix = np.zeros((n_test, n_train))
    
    for col in X_train.columns:
        col_data_train = X_train[col]
        col_data_test = X_test[col]
        
        if pd.api.types.is_numeric_dtype(col_data_train):
            # Numeric column: range-normalized Manhattan distance
            col_min = min(col_data_train.min(), col_data_test.min())
            col_max = max(col_data_train.max(), col_data_test.max())
            col_range = col_max - col_min
            
            if col_range == 0:
                col_range = 1
            
            for i in range(n_test):
                for j in range(n_train):
                    dist_matrix[i, j] += abs(col_data_test.iloc[i] - col_data_train.iloc[j]) / col_range
        else:
            # Categorical column: 0 if same, 1 if different
            for i in range(n_test):
                for j in range(n_train):
                    if col_data_test.iloc[i] != col_data_train.iloc[j]:
                        dist_matrix[i, j] += 1
    
    # Normalize by number of columns
    n_cols = len(X_train.columns)
    dist_matrix = dist_matrix / n_cols
    
    return dist_matrix

def majority_vote_weighted(y_neighbors, distances, epsilon=1e-5):
    weights = 1 / (distances + epsilon)
    classes = np.unique(y_neighbors)
    class_weights = np.array([weights[y_neighbors == c].sum() for c in classes])
    
    if len(class_weights) == 0 or np.all(np.isnan(class_weights)):
        return np.random.choice(y_neighbors), False
    
    max_weight = np.nanmax(class_weights)
    candidates = classes[class_weights == max_weight]
    return np.random.choice(candidates), len(candidates) > 1

def evaluate_knn(X_train, y_train, X_test, y_test, k_values, metric='gower', is_binary=False):
    results = []
    
    if metric == 'gower':
        X_train_prep = prepare_for_gower(X_train)
        X_test_prep = prepare_for_gower(X_test)
        dist_matrix = gower_distance_matrix(X_train_prep, X_test_prep)
    else:
        if is_binary:
            X_test_input = X_test.astype(bool)
            X_train_input = X_train.astype(bool)
        else:
            X_test_input = X_test.values
            X_train_input = X_train.values
        dist_matrix = pairwise_distances(X_test_input, X_train_input, metric=metric)
    
    for k in k_values:
        if k > len(X_train):
            continue
        indices = np.argsort(dist_matrix, axis=1)[:, :k]
        predictions = []
        
        for i in range(len(X_test)):
            neighbor_vals = y_train.iloc[indices[i]].values
            neighbor_dists = dist_matrix[i, indices[i]]
            pred, _ = majority_vote_weighted(neighbor_vals, neighbor_dists)
            predictions.append(pred)
        
        results.append({'k': k, 'accuracy': accuracy_score(y_test, predictions)})
    
    return results

# ==========================================================
# CONFIGURATION
# ==========================================================

class Config:
    R = 'Y'              # 'Y' for randomizzazione, 'N' for no
    PR = 40              # Randomization percentage: 0, 10, 20, 30, 40, 100
    SYNTHESIZER_TYPE = 'CTGAN'
    
    RANDOMIZATION_LABEL = f'RY_PR{PR}' if R == 'Y' else 'RN_PR0'
    
    # ONYXIA paths
    BASE_PATH = rf"/home/onyxia/work/data/"
    
    INPUT_REAL = f'{BASE_PATH}/Step1/Output'
    INPUT_SYNTH = f'{BASE_PATH}/Step2/Output'
    OUTPUT_REPORT = f'{BASE_PATH}/Step4/Output/Report/attack1_used'
    OUTPUT_PLOT = f'{BASE_PATH}/Step4/Output/Plot'
    
    K_VALUES = [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15,
        17, 19, 21, 23, 25, 27, 29, 31, 35, 39, 43, 47,
        51, 59, 67, 75, 83, 99, 115, 131, 147, 163, 179, 195
    ]
    
    METRICS = [
        'l2', 'sqeuclidean', 'manhattan', 'l1', 'cityblock',
        'cosine', 'braycurtis', 'nan_euclidean', 'correlation', 
        'hamming', 'euclidean', 'canberra'
    ]
    
    BINARY_METRICS = ['sokalmichener', 'sokalsneath', 'yule', 'russellrao', 'dice', 'rogerstanimoto', 'jaccard']

SYNTHESIZER_TYPES = ['CTGAN', 'TVAE', 'XGBoost_O0']

# ==========================================================
# COLOR PALETTE (13 colors for 13 models)
# ==========================================================

COLOR_PALETTE = [
    '#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E',
    '#BC4A6C', '#1C7C54', '#D62828', '#003D5B', '#E09F3E',
    '#7B2CBF', '#F72585', '#4CC9F0',
]

MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'H', 'd', 'p']
LINESTYLES = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (1, 1))]

# ==========================================================
# SHARED STYLE SETUP
# ==========================================================

def setup_plot_style():
    plt.rcParams.update({
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
        'font.size': 12,
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 11,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'ps.useafm': True,
        'pdf.use14corefonts': True,
        'text.usetex': False,
    })

# ==========================================================
# DATA LOADING (senza comuni)
# ==========================================================

def load_data(cfg):
    real_path = f'{cfg.INPUT_REAL}/real_data_datasetM10_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2.csv'
    unseen_path = f'{cfg.INPUT_REAL}/real_data_datasetM2_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2.csv'
    synth_path = f'{cfg.INPUT_SYNTH}/synthetic_data_datasetM10_TH20_R{cfg.R}_PR{cfg.PR}_4CAT_MISS_X2_{cfg.SYNTHESIZER_TYPE}.csv'
    
    print(f"Loading real data from: {real_path}")
    print(f"Loading unseen data from: {unseen_path}")
    print(f"Loading synth data from: {synth_path}")
    
    real = pd.read_csv(real_path)
    unseen = pd.read_csv(unseen_path)
    synth = pd.read_csv(synth_path)
    
    # Drop unnecessary columns
    cols_to_drop = ['id', 'score1', 'score2', 'bin1', 'bin2', 'raw_class', 
                    'age_norm', 'activity_norm', 'predisposition_norm', 
                    'birth_year', 'birth_month', 'birth_day', 
                    'municipality_birth', 'birth_dayofyear']
    
    for df in [real, unseen, synth]:
        cols_exist = [col for col in cols_to_drop if col in df.columns]
        if cols_exist:
            df.drop(columns=cols_exist, inplace=True)
    
    return real, unseen, synth

# ==========================================================
# BUILD FIXED ORDERING
# ==========================================================

def build_fixed_model_order_from_reports(all_reports, model_col='model', top_n=13):
    combined = pd.concat(all_reports, ignore_index=True)
    max_per_model = combined.groupby(model_col)['acc_real'].max()
    max_per_model = max_per_model.sort_values(ascending=False)
    
    if top_n is not None and top_n < len(max_per_model):
        top_models = max_per_model.head(top_n).index.tolist()
    else:
        top_models = max_per_model.index.tolist()
    
    model_labels = [m.replace('MM_', '').replace('GOWER', 'Gower') for m in top_models]
    return top_models, model_labels

# ==========================================================
# SAVE SHARED LEGEND
# ==========================================================

def save_shared_legend(top_models, model_labels, cfg, plot_path):
    setup_plot_style()
    fig, ax = plt.subplots(figsize=(1, 1))

    for idx, label in enumerate(model_labels):
        ax.plot([], [], marker=MARKERS[idx % len(MARKERS)],
                linestyle=LINESTYLES[idx % len(LINESTYLES)],
                color=COLOR_PALETTE[idx], label=label,
                linewidth=2, markersize=6,
                markeredgecolor='black', markeredgewidth=0.8)

    handles, labels = ax.get_legend_handles_labels()
    plt.close(fig)

    n_models = len(model_labels)
    n_cols = min(7, n_models)
    
    fig_leg = plt.figure(figsize=(14, 1.5))
    fig_leg.legend(handles, labels, loc='center', ncol=n_cols,
                   frameon=False, fontsize=10, handlelength=2.0, columnspacing=1.2)
    plt.axis('off')

    legend_eps = plot_path / f'legend_all_metrics_{cfg.RANDOMIZATION_LABEL}.eps'
    legend_jpeg = legend_eps.with_suffix('.jpeg')

    plt.savefig(legend_eps, format='eps', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(legend_jpeg, format='jpeg', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig_leg)
    print(f'  ✓ SHARED LEGEND EPS → {legend_eps.name}')
    print(f'  ✓ SHARED LEGEND JPEG → {legend_jpeg.name}')

# ==========================================================
# PLOT FUNCTION
# ==========================================================

def plot_top_models_accuracy(results_df, cfg, plot_path, top_models, model_labels):
    setup_plot_style()
    results_model_col = 'model' if 'model' in results_df.columns else 'modello'

    for metric, ylabel, suffix in [('acc_real', 'TVR Accuracy', 'TVR'),
                                   ('acc_unseen', 'BVR Accuracy', 'BVR')]:
        fig, ax = plt.subplots(figsize=(11, 7))
        ax.set_facecolor('white')

        for idx, model in enumerate(top_models):
            if model not in results_df[results_model_col].values:
                continue
            df_m = results_df[results_df[results_model_col] == model].sort_values('k')
            if len(df_m) == 0:
                continue
            ax.plot(df_m['k'], df_m[metric],
                    marker=MARKERS[idx % len(MARKERS)],
                    linestyle=LINESTYLES[idx % len(LINESTYLES)],
                    color=COLOR_PALETTE[idx],
                    linewidth=2, markersize=6,
                    markeredgecolor='black', markeredgewidth=0.8)

        ax.set_xlabel(r'$k$', fontsize=14, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=14, fontweight='bold')
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0.25, top=0.55)
        ax.grid(True, alpha=0.3, linestyle='--')

        plt.tight_layout()
        out_eps = plot_path / f'accuracy_vs_k_{suffix}_{cfg.SYNTHESIZER_TYPE}_{cfg.RANDOMIZATION_LABEL}.eps'
        out_jpeg = out_eps.with_suffix('.jpeg')

        plt.savefig(out_eps, format='eps', dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(out_jpeg, format='jpeg', dpi=150, bbox_inches='tight', facecolor='white')
        plt.close()
        print(f'  ✓ EPS → {out_eps.name}')
        print(f'  ✓ JPEG → {out_jpeg.name}')

# ==========================================================
# RUN PIPELINE FOR ONE SYNTHESIZER
# ==========================================================

def run_pipeline_for_synthesizer(cfg, synth_type):
    cfg.SYNTHESIZER_TYPE = synth_type
    
    print(f"\n{'='*60}")
    print(f"Processing synthesizer: {synth_type}")
    print(f"{'='*60}")
    
    report_path = Path(cfg.OUTPUT_REPORT)
    report_path.mkdir(parents=True, exist_ok=True)
    
    # Load data
    df_real, df_unseen, df_synth = load_data(cfg)
    
    X_synth, y_synth = df_synth.drop(columns=['diagnosis']), df_synth['diagnosis']
    X_real, y_real = df_real.drop(columns=['diagnosis']), df_real['diagnosis']
    X_unseen, y_unseen = df_unseen.drop(columns=['diagnosis']), df_unseen['diagnosis']
    
    # Label encoding for non-Gower metrics
    encoders = {}
    for col in X_synth.select_dtypes(include=['object']).columns:
        le = LabelEncoder()
        X_synth[col] = le.fit_transform(X_synth[col].astype(str))
        encoders[col] = le
    
    for df in [X_real, X_unseen]:
        for col in encoders:
            df[col] = df[col].map(lambda x: encoders[col].transform([str(x)])[0] if str(x) in encoders[col].classes_ else -1)
    
    all_results = []
    
    # GOWER evaluation
    print("Evaluating GOWER...")
    gower_real = evaluate_knn(X_synth, y_synth, X_real, y_real, cfg.K_VALUES, metric='gower')
    gower_unseen = evaluate_knn(X_synth, y_synth, X_unseen, y_unseen, cfg.K_VALUES, metric='gower')
    
    gower_dict_real = {r['k']: r['accuracy'] for r in gower_real}
    gower_dict_unseen = {r['k']: r['accuracy'] for r in gower_unseen}
    
    for k in cfg.K_VALUES:
        if k in gower_dict_real and k in gower_dict_unseen:
            all_results.append({
                'model': 'GOWER',
                'k': k,
                'acc_real': gower_dict_real[k],
                'acc_unseen': gower_dict_unseen[k],
                'delta': abs(gower_dict_real[k] - gower_dict_unseen[k])
            })
    
    # Multi-metric evaluation
    for metric in cfg.METRICS:
        print(f"Evaluating {metric}...")
        is_binary = metric in cfg.BINARY_METRICS
        
        real_res = evaluate_knn(X_synth, y_synth, X_real, y_real, cfg.K_VALUES, metric=metric, is_binary=is_binary)
        unseen_res = evaluate_knn(X_synth, y_synth, X_unseen, y_unseen, cfg.K_VALUES, metric=metric, is_binary=is_binary)
        
        real_dict = {r['k']: r['accuracy'] for r in real_res}
        unseen_dict = {r['k']: r['accuracy'] for r in unseen_res}
        
        for k in cfg.K_VALUES:
            if k in real_dict and k in unseen_dict:
                all_results.append({
                    'model': f'MM_{metric}',
                    'k': k,
                    'acc_real': real_dict[k],
                    'acc_unseen': unseen_dict[k],
                    'delta': abs(real_dict[k] - unseen_dict[k])
                })
    
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(report_path / f'report_unificato_completo_{cfg.RANDOMIZATION_LABEL}_{synth_type}.csv', index=False)
    print(f"  ✓ Results saved for {synth_type}")
    
    return results_df

# ==========================================================
# MAIN
# ==========================================================

def generate_plots_only():
    cfg = Config()
    
    report_path = Path(cfg.OUTPUT_REPORT)
    plot_path = Path(cfg.OUTPUT_PLOT)
    report_path.mkdir(parents=True, exist_ok=True)
    plot_path.mkdir(parents=True, exist_ok=True)
    
    print(f"Pipeline started")
    print(f"  Randomization: {cfg.RANDOMIZATION_LABEL}")
    print(f"  K values: {len(cfg.K_VALUES)} (max {max(cfg.K_VALUES)})")
    print(f"  Metrics: {len(cfg.METRICS)} + GOWER")
    print(f"  Report output: {report_path}")
    print(f"  Plot output: {plot_path}")
    print("-" * 60)
    
    # Run evaluation for all synthesizers
    all_reports = []
    loaded_data = {}
    
    for synth in SYNTHESIZER_TYPES:
        results_df = run_pipeline_for_synthesizer(cfg, synth)
        all_reports.append(results_df)
        loaded_data[synth] = results_df
    
    # Build fixed model ordering
    print("\n" + "=" * 60)
    print("Building fixed model ordering from all synthesizers")
    print("=" * 60)
    
    top_models, model_labels = build_fixed_model_order_from_reports(all_reports, top_n=13)
    
    print(f'\nFixed model order ({len(top_models)} models):')
    for i, (m, l) in enumerate(zip(top_models, model_labels)):
        print(f'  {i+1:2d}. {l}  ({m})')
    
    # Save shared legend
    print()
    save_shared_legend(top_models, model_labels, cfg, plot_path)
    
    # Generate plots
    print()
    print("=" * 60)
    print("Generating plots with fixed legend")
    print("=" * 60)
    
    for synth, results_df in loaded_data.items():
        cfg.SYNTHESIZER_TYPE = synth
        print(f'\n--- {synth} ---')
        plot_top_models_accuracy(results_df, cfg, plot_path, top_models, model_labels)
    
    print()
    print(f"All EPS + JPEG saved to: {plot_path}")
    print("=" * 60)
    
    return loaded_data

# ==========================================================
# ENTRY POINT
# ==========================================================

if __name__ == '__main__':
    start = time.time()
    all_results = generate_plots_only()
    print(f"\nTotal execution time: {time.time() - start:.1f}s")

Pipeline started
  Randomization: RY_PR40
  K values: 39 (max 195)
  Metrics: 12 + GOWER
  Report output: /home/onyxia/work/data/Step4/Output/Report/attack1_used
  Plot output: /home/onyxia/work/data/Step4/Output/Plot
------------------------------------------------------------

Processing synthesizer: CTGAN
Loading real data from: /home/onyxia/work/data//Step1/Output/real_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2.csv
Loading unseen data from: /home/onyxia/work/data//Step1/Output/real_data_datasetM2_TH20_RY_PR40_4CAT_MISS_X2.csv
Loading synth data from: /home/onyxia/work/data//Step2/Output/synthetic_data_datasetM10_TH20_RY_PR40_4CAT_MISS_X2_CTGAN.csv
Evaluating GOWER...
